# 🎬 HyperFlow H3 Flash Ref2VA INT8 Fresh Studio Suite
### Model: `VideoRebirthAI/hyperflow-h3-flash` (ref2va_int8_convrot_comfyui)
### Target Checkpoint: `minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors` (15.3 GB)
### Text Encoder: `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors` (14.6 GB)
### Video VAE: `minimax_h3_video_vae_int8_convrot.safetensors` | Audio VAE: `minimax_h3_audio_vae_fp32.safetensors`
### Attached Dataset: `minimax-h3-open-weights`
### Execution Order:
1. **Test 1: Official GitHub Repo Benchmark** (Official author workflow & sample reference images from VideoRebirth)
2. **Test 2: Handheld Vlog** (Single creator talking into camera, conversational lip-sync, handheld motion)
3. **Test 3: Cinematic Wide Shot** (Atmospheric sweeping landscape, mountain temple, environmental soundscape)
4. **Test 4: 2-Person Multishot Dialogue** (Dramatic confrontation, strict speaker isolation, closed lips)
5. **Test 5: 2-Person Fast-Motion Fight Scene** (Martial arts action, dodge & strike, synced impact audio)


In [ ]:
# [Step 1/5] Setup Environment, Dependencies & ComfyUI
import os
import sys
import subprocess
import time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

print("=" * 75, flush=True)
print(">>> [1/5] Installing Dependencies, ComfyUI, and ComfyUI-HyperFlow-Flash...", flush=True)
print("=" * 75, flush=True)

# 1. Install Cloudflared for remote monitoring
subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    shell=True,
    check=True
)
print("✓ Cloudflared binary installed.", flush=True)

# 2. Clone ComfyUI core
if not os.path.exists("ComfyUI"):
    subprocess.run("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git", shell=True, check=True)
    print("✓ Cloned ComfyUI core repository.", flush=True)

# 3. Clone ComfyUI-HyperFlow-Flash
custom_nodes_path = "ComfyUI/custom_nodes/ComfyUI-HyperFlow-Flash"
if not os.path.exists(custom_nodes_path):
    subprocess.run(
        "git clone --depth 1 https://github.com/Video-Rebirth/ComfyUI-HyperFlow-Flash.git " + custom_nodes_path,
        shell=True,
        check=True
    )
    print("✓ Cloned ComfyUI-HyperFlow-Flash custom node.", flush=True)

# 4. Install requirements
pkgs = [
    "huggingface_hub>=0.28.0",
    "hf_transfer",
    "gguf",
    "pillow",
    "requests",
    "tqdm"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "ComfyUI/requirements.txt"], check=True)
print("✓ Python requirements installed successfully.", flush=True)


In [ ]:
# [Step 2/5] High-Speed Model Ingestion
import os
import shutil
import time
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("=" * 75, flush=True)
print(">>> [2/5] Provisioning Model Weights (Attached Datasets + HF Transfer)...", flush=True)
print("=" * 75, flush=True)

os.makedirs("ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("ComfyUI/models/vae", exist_ok=True)

import glob
ds_candidates = glob.glob('/kaggle/input/*minimax*weights*')
ds_root = ds_candidates[0] if ds_candidates else '/kaggle/input/minimax-h3-open-weights'

# 1. Provision Text Encoder
clip_dst = "ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
src_clip = os.path.join(ds_root, "clip/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors")
if os.path.exists(src_clip):
    if not os.path.exists(clip_dst):
        try:
            os.symlink(src_clip, clip_dst)
        except Exception:
            shutil.copyfile(src_clip, clip_dst)
    print(f"✓ Linked text encoder from mounted dataset: {clip_dst}", flush=True)
else:
    print("Downloading text encoder from Hugging Face...", flush=True)
    p = hf_hub_download(repo_id="Comfy-Org/MiniMax-H3", filename="text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors")
    if os.path.exists(clip_dst):
        os.remove(clip_dst)
    os.symlink(p, clip_dst)
    print("✓ Downloaded text encoder.", flush=True)

# 2. Provision Video VAE
vvae_dst = "ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors"
src_vvae = os.path.join(ds_root, "vae/minimax_h3_video_vae_int8_convrot.safetensors")
if os.path.exists(src_vvae):
    if not os.path.exists(vvae_dst):
        try:
            os.symlink(src_vvae, vvae_dst)
        except Exception:
            shutil.copyfile(src_vvae, vvae_dst)
    print(f"✓ Linked video VAE from mounted dataset: {vvae_dst}", flush=True)
else:
    print("Downloading video VAE from Hugging Face...", flush=True)
    p = hf_hub_download(repo_id="Comfy-Org/MiniMax-H3", filename="vae/minimax_h3_video_vae_fp16.safetensors")
    if os.path.exists(vvae_dst):
        os.remove(vvae_dst)
    os.symlink(p, vvae_dst)
    print("✓ Downloaded video VAE.", flush=True)

# 3. Provision Audio VAE
avae_dst = "ComfyUI/models/vae/minimax_h3_audio_vae_fp32.safetensors"
src_avae = os.path.join(ds_root, "vae/minimax_h3_audio_vae_fp32.safetensors")
if os.path.exists(src_avae):
    if not os.path.exists(avae_dst):
        try:
            os.symlink(src_avae, avae_dst)
        except Exception:
            shutil.copyfile(src_avae, avae_dst)
    print(f"✓ Linked audio VAE from mounted dataset: {avae_dst}", flush=True)
else:
    print("Downloading audio VAE from Hugging Face...", flush=True)
    p = hf_hub_download(repo_id="Comfy-Org/MiniMax-H3", filename="vae/minimax_h3_audio_vae_fp32.safetensors")
    if os.path.exists(avae_dst):
        os.remove(avae_dst)
    os.symlink(p, avae_dst)
    print("✓ Downloaded audio VAE.", flush=True)

# 4. Download HyperFlow H3 Flash Ref2VA INT8 Convrot Safetensors
unet_dst = "ComfyUI/models/diffusion_models/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors"
if not os.path.exists(unet_dst) or os.path.getsize(unet_dst) < 100 * 1024 * 1024:
    print("Downloading HyperFlow H3 Flash Ref2VA INT8 from VideoRebirthAI/hyperflow-h3-flash...", flush=True)
    t0 = time.time()
    p = hf_hub_download(
        repo_id="VideoRebirthAI/hyperflow-h3-flash",
        filename="ref2va_int8_convrot_comfyui/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors"
    )
    if os.path.exists(unet_dst):
        os.remove(unet_dst)
    try:
        os.link(p, unet_dst)
    except Exception:
        try:
            os.symlink(p, unet_dst)
        except Exception:
            shutil.copyfile(p, unet_dst)
    dur = time.time() - t0
    size_gb = os.path.getsize(unet_dst) / (1024 ** 3)
    print(f"✓ Ready: {unet_dst} ({size_gb:.2f} GB) in {dur:.1f}s", flush=True)
else:
    print(f"✓ Target transformer already present: {unet_dst}", flush=True)

print("✓ All 4 checkpoints ready in ComfyUI/models/.", flush=True)


In [ ]:
# [Step 3/5] Ingest Official Example References & Generate Test Assets
import os
import shutil
import urllib.request
from PIL import Image, ImageDraw

input_dir = "ComfyUI/input"
os.makedirs(input_dir, exist_ok=True)

print("=" * 75, flush=True)
print(">>> [3/5] Ingesting Official Example Images & Generating Test Assets...", flush=True)
print("=" * 75, flush=True)

# 1. Download official reference images directly from the GitHub repository
official_urls = {
    "ref_image_1.png": "https://raw.githubusercontent.com/Video-Rebirth/hyperflow-h3-flash/main/integrations/ComfyUI-HyperFlow-Flash/example_workflows/hyperflow_ref2v_int8.jpg",
    "ref_image_2.png": "https://raw.githubusercontent.com/Video-Rebirth/hyperflow-h3-flash/main/integrations/ComfyUI-HyperFlow-Flash/example_workflows/hyperflow_i2v_int8.jpg",
    "hyperflow_flf2v_int8.jpg": "https://raw.githubusercontent.com/Video-Rebirth/hyperflow-h3-flash/main/integrations/ComfyUI-HyperFlow-Flash/example_workflows/hyperflow_flf2v_int8.jpg"
}

for fname, url in official_urls.items():
    dst = os.path.join(input_dir, fname)
    try:
        urllib.request.urlretrieve(url, dst)
        print(f"✓ Downloaded official reference: {fname}", flush=True)
    except Exception as e:
        print(f"! Notice on {fname}: {e}", flush=True)

# Also copy from cloned repo folder if available
ex_dir = "ComfyUI/custom_nodes/ComfyUI-HyperFlow-Flash/example_workflows"
if os.path.exists(ex_dir):
    for f in os.listdir(ex_dir):
        if f.endswith(".jpg") or f.endswith(".png"):
            src_f = os.path.join(ex_dir, f)
            dst_f = os.path.join(input_dir, f)
            if not os.path.exists(dst_f):
                shutil.copyfile(src_f, dst_f)
                print(f"✓ Copied asset from repo: {f}", flush=True)

# 2. Generate customized reference assets for scenarios
def create_reference_canvas(filename, bg_color, accent_color, title_text, char_type):
    img = Image.new("RGB", (768, 768), color=bg_color)
    draw = ImageDraw.Draw(img)
    cx, cy = 384, 384
    draw.ellipse([cx - 120, cy - 200, cx + 120, cy + 40], fill=accent_color)
    draw.ellipse([cx - 180, cy + 30, cx + 180, cy + 380], fill=accent_color)
    draw.ellipse([cx - 90, cy - 160, cx + 90, cy + 10], fill=(235, 195, 165))
    draw.rectangle([20, 20, 748, 80], fill=(20, 20, 20))
    draw.text((40, 40), f"REF: {title_text} [{char_type}]", fill=(255, 255, 255))
    path = os.path.join(input_dir, filename)
    img.save(path)
    print(f"✓ Created reference asset: {path}", flush=True)

create_reference_canvas("vlog_creator.png", (40, 50, 65), (70, 90, 120), "VLOG CREATOR", "Urban Casual")
create_reference_canvas("wide_temple.png", (25, 45, 30), (50, 80, 60), "MOUNTAIN TEMPLE", "Landscape Setting")
create_reference_canvas("char_detective.png", (30, 30, 45), (60, 60, 80), "SUBJECT 1 DETECTIVE", "Trench Coat")
create_reference_canvas("char_informant.png", (45, 30, 30), (90, 60, 60), "SUBJECT 2 INFORMANT", "Dark Hoodie")
create_reference_canvas("fighter_red.png", (60, 20, 20), (120, 40, 40), "FIGHTER 1 RED", "Combat Stance")
create_reference_canvas("fighter_black.png", (20, 20, 25), (45, 45, 55), "FIGHTER 2 BLACK", "Martial Arts")

print("✓ All test reference images ready in ComfyUI/input/", flush=True)


In [ ]:
# [Step 4/5] Launch Headless ComfyUI Server & Cloudflare Tunnel
import subprocess
import threading
import time
import re
import urllib.request
import json

PORT = 8188

print("=" * 75, flush=True)
print(">>> [4/5] Starting ComfyUI Server on port 8188...", flush=True)
print("=" * 75, flush=True)

comfy_cmd = f"{sys.executable} ComfyUI/main.py --listen 127.0.0.1 --port {PORT} --enable-cors-header --preview-method none"
comfy_proc = subprocess.Popen(
    comfy_cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

def monitor_comfy():
    for line in comfy_proc.stdout:
        if any(w in line for w in ["Loaded", "Prompt", "Executing", "HyperFlow", "error", "Error", "CUDA"]):
            print(f"[ComfyUI] {line.strip()}", flush=True)

threading.Thread(target=monitor_comfy, daemon=True).start()

# Launch Cloudflared tunnel for live monitoring
print(">>> Launching Cloudflare Tunnel for live browser inspection...", flush=True)
tunnel_proc = subprocess.Popen(
    f"/usr/local/bin/cloudflared tunnel --url http://127.0.0.1:{PORT}",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
t_start = time.time()
while time.time() - t_start < 40:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

print("\n" + "=" * 80, flush=True)
if public_url:
    print(f"🌐 COMFYUI LIVE WEB UI: {public_url}", flush=True)
print("=" * 80 + "\n", flush=True)

server_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/system_stats", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                print("✓ ComfyUI API is fully initialized and ready for automated jobs!", flush=True)
                break
    except Exception:
        time.sleep(2)

if not server_ready:
    raise RuntimeError("ComfyUI server failed to start within the timeout period.")


In [ ]:
# [Step 5/5] Automated End-to-End Batch Test Runner
import urllib.request
import json
import time
import shutil

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 75, flush=True)
print(">>> [5/5] Executing the Combined Test Suite via ComfyUI API...", flush=True)
print("=" * 75, flush=True)

def build_workflow(prompt_text, ref_images, output_prefix, seed=42):
    wf = {
        "1": {
            "class_type": "HyperFlowLoader",
            "inputs": {
                "unet_name": "minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors"
            }
        },
        "2": {
            "class_type": "CLIPLoader",
            "inputs": {
                "clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
                "type": "minimax",
                "device": "default"
            }
        },
        "3": {
            "class_type": "VAELoader",
            "inputs": {
                "vae_name": "minimax_h3_video_vae_fp16.safetensors"
            }
        },
        "4": {
            "class_type": "VAELoader",
            "inputs": {
                "vae_name": "minimax_h3_audio_vae_fp32.safetensors"
            }
        },
        "5": {
            "class_type": "MiniMaxH3ReferenceToVideo",
            "inputs": {
                "clip": ["2", 0],
                "vae": ["3", 0],
                "audio_vae": ["4", 0],
                "prompt": prompt_text,
                "width": 1344,
                "height": 768,
                "length": 124,
                "ref_image_size": "match"
            }
        },
        "6": {
            "class_type": "HyperFlowSigmas",
            "inputs": {
                "model": ["1", 0],
                "steps": 6
            }
        },
        "7": {
            "class_type": "KSamplerSelect",
            "inputs": {
                "sampler_name": "euler"
            }
        },
        "8": {
            "class_type": "RandomNoise",
            "inputs": {
                "noise_seed": seed
            }
        },
        "9": {
            "class_type": "BasicGuider",
            "inputs": {
                "model": ["1", 0],
                "conditioning": ["5", 0]
            }
        },
        "10": {
            "class_type": "SamplerCustomAdvanced",
            "inputs": {
                "noise": ["8", 0],
                "guider": ["9", 0],
                "sampler": ["7", 0],
                "sigmas": ["6", 0],
                "latent_image": ["5", 1]
            }
        },
        "11": {
            "class_type": "VAEDecode",
            "inputs": {
                "samples": ["10", 0],
                "vae": ["3", 0]
            }
        },
        "12": {
            "class_type": "VAEDecodeAudio",
            "inputs": {
                "samples": ["10", 0],
                "vae": ["4", 0]
            }
        },
        "13": {
            "class_type": "CreateVideo",
            "inputs": {
                "images": ["11", 0],
                "audio": ["12", 0],
                "fps": 24
            }
        },
        "14": {
            "class_type": "SaveVideo",
            "inputs": {
                "video": ["13", 0],
                "filename_prefix": output_prefix,
                "format": "video/h264-mp4"
            }
        }
    }

    for i, ref_img in enumerate(ref_images):
        node_id = str(20 + i)
        wf[node_id] = {
            "class_type": "LoadImage",
            "inputs": {"image": ref_img}
        }
        wf["5"]["inputs"][f"ref_images.ref_image_{i}"] = [node_id, 0]

    return wf

def run_test_case(test_idx, test_name, prompt, ref_images, seed=100):
    print("\n" + "=" * 75, flush=True)
    print(f"▶ RUNNING TEST CASE {test_idx}: {test_name}", flush=True)
    print("=" * 75, flush=True)
    print(f"Prompt: {prompt}", flush=True)
    print(f"References: {ref_images}", flush=True)

    prefix = f"test_{test_idx}_{test_name.lower().replace(' ', '_')}"
    workflow = build_workflow(prompt, ref_images, prefix, seed=seed)

    payload = json.dumps({"prompt": workflow}).encode("utf-8")
    req = urllib.request.Request(
        "http://127.0.0.1:8188/prompt",
        data=payload,
        headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req) as resp:
        res = json.loads(resp.read().decode("utf-8"))
        prompt_id = res.get("prompt_id")
        print(f"✓ Enqueued job prompt_id: {prompt_id}", flush=True)

    t0 = time.time()
    completed = False
    while time.time() - t0 < 900:
        time.sleep(10)
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}") as h_resp:
                hist = json.loads(h_resp.read().decode("utf-8"))
                if prompt_id in hist:
                    status = hist[prompt_id].get("status", {})
                    if status.get("completed", False):
                        completed = True
                        break
        except Exception:
            pass

    elapsed = time.time() - t0
    if not completed:
        print(f"✗ Test Case {test_idx} timed out or failed!", flush=True)
        return None

    print(f"✓ Test Case {test_idx} completed in {elapsed:.1f}s!", flush=True)

    comfy_out_dir = "ComfyUI/output"
    matched_video = None
    for root, _, files in os.walk(comfy_out_dir):
        for f in files:
            if f.startswith(prefix) and f.endswith(".mp4"):
                src_file = os.path.join(root, f)
                dst_file = os.path.join(OUTPUT_DIR, f"{prefix}.mp4")
                shutil.copyfile(src_file, dst_file)
                matched_video = dst_file
                break

    if matched_video:
        size_mb = os.path.getsize(matched_video) / (1024 ** 2)
        print(f"🎉 Output Video Saved: {matched_video} ({size_mb:.2f} MB)", flush=True)
    return matched_video

# ==============================================================================
# THE 5 COMBINED TEST SUITE SCENARIOS (GITHUB REPO TEST FIRST)
# ==============================================================================
tests = [
    {
        "idx": 1,
        "name": "official_repo_ref2v_walkthrough",
        "prompt": "<Picture 1> walks slowly through the scene of <Picture 2>, looks around and smiles; the camera follows at shoulder height, footsteps and soft ambient sound fill the space.",
        "ref_images": ["ref_image_1.png", "ref_image_2.png"],
        "seed": 777
    },
    {
        "idx": 2,
        "name": "handheld_vlog",
        "prompt": "<Picture 1> holds a smartphone camera in front of him for an energetic daily vlog on a sunlit urban street, talking directly into the lens with natural hand gestures and organic handheld camera micro-shake: <d>[English] What is up everyone! Today we are testing this brand new joint video and audio generation model for the first time, and the lip sync is unbelievable.</d> City traffic ambiance, soft passing pedestrians, crisp outdoor vocal acoustics.",
        "ref_images": ["vlog_creator.png"],
        "seed": 42
    },
    {
        "idx": 3,
        "name": "cinematic_wide_shot",
        "prompt": "<Picture 1> A breathtaking cinematic extreme wide shot slowly sweeping over an ancient misty temple nestled in towering pine-covered mountain peaks at sunrise. Golden morning light cuts through drifting fog layers as distant birds soar across the valley. Gentle mountain breeze rustles the trees, distant temple bell resonance, deep atmospheric ambient soundscape.",
        "ref_images": ["wide_temple.png"],
        "seed": 101
    },
    {
        "idx": 4,
        "name": "two_person_multishot",
        "prompt": "subject_definitions: <Subject 1> is the detective in <Picture 1>. <Subject 2> is the informant in <Picture 2>. detailed_description: In a dim rainy neon alleyway, <Subject 1> steps forward and speaks sternly to <Subject 2>: <d>[English] You told me this drop was clean, so why are the feds already on the block?</d> <Subject 2> does not speak and his lips remain completely closed, nervously glancing sideways. overall_soundscape: Torrential rain splashing on asphalt, distant police siren echoing through the city.",
        "ref_images": ["char_detective.png", "char_informant.png"],
        "seed": 202
    },
    {
        "idx": 5,
        "name": "fast_motion_fight_scene",
        "prompt": "Dynamic high-speed martial arts combat in an industrial warehouse between <Picture 1> and <Picture 2>. Fighter 1 delivers a rapid high roundhouse kick which Fighter 2 ducks under and retaliates with a swift double palm strike. The camera vigorously tracks and whip-pans with the action. Fast whooshes of punches cutting through the air, heavy physical impact thuds, squeaking combat boots on concrete, intense exertion grunts: <d>[English] Hah! ... Oof!</d>",
        "ref_images": ["fighter_red.png", "fighter_black.png"],
        "seed": 303
    }
]

results = []
for t in tests:
    res = run_test_case(t["idx"], t["name"], t["prompt"], t["ref_images"], t["seed"])
    results.append({"test": t["name"], "path": res})

print("\n" + "=" * 80, flush=True)
print("🏁 COMPLETE TEST SUITE FINISHED! SUMMARY:", flush=True)
for r in results:
    print(f" - {r['test']}: {r['path']}", flush=True)
print("=" * 80 + "\n", flush=True)

counter = 0
while True:
    time.sleep(60)
    counter += 1
    if counter % 10 == 0:
        print(f"[Heartbeat] All 5 tests complete. Runner online ({counter}m).", flush=True)
